<a href="https://colab.research.google.com/github/Regge12/Text-classification-Positive-or-Negative-/blob/main/Text_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import re
import json
import string

In [13]:
# Stop words: high-frequency words with no sentiment value
# Removing them reduces noise and speeds up pattern matching
STOPWORDS = {
    "the", "a", "an", "is", "it", "this", "that", "was", "and",
    "of", "to", "in", "i", "my", "for", "with", "on", "at", "be",
    "are", "have", "had", "but", "or", "as", "so", "we", "he",
    "she", "they", "its", "by", "from", "about", "been", "one",
    "all", "if", "when", "there", "their", "can", "what", "more",
    "some", "then", "than", "him", "her", "who", "which", "you"
}

# Negation words: when detected, the sentiment of the next matched token is flipped
NEGATION_WORDS = {"not", "never", "no", "hardly", "barely", "neither", "nor"}

# POSITIVE PATTERNS
# Each pattern uses re.fullmatch so the entire token must match
# Using the stem of the word captures variants like r"enjoy\w*" matches enjoyed, enjoyable
POSITIVE_PATTERNS = [
    r"enjoy\w*",
    r"terrif\w*",
    r"brillian\w*",
    r"beauti\w*",
    r"great\w*",
    r"outstand\w*",
    r"excel\w*",
    r"perfect\w*",
    r"love\w*",
    r"amaz\w*",
    r"wonderf\w*",
    r"masterpi\w*",
    r"impress\w*",
    r"captivat\w*",
    r"heartwarming",
    r"recommend\w*",
    r"inspir\w*",
    r"hilar\w*",
    r"delight\w*",
    r"superb\w*",
    r"charm\w*",
    r"entertain\w*",
    r"engag\w*",
    r"memorabl\w*",
    r"powerf\w*",
    r"touch\w*",
    r"genuin\w*",
    r"witty\w*",
    r"good\w*",
    r"fun\w*",
    r"strong\w*",
    r"stylish\w*",
    r"sincere\w*",
    r"vivid\w*",
]

# NEGATIVE PATTERNS
NEGATIVE_PATTERNS = [
    r"terribl\w*",
    r"awful\w*",
    r"bor\w*",
    r"disappoint\w*",
    r"worst\w*",
    r"bad\w*",
    r"dull\w*",
    r"poor\w*",
    r"painf\w*",
    r"horribl\w*",
    r"horrif\w*",
    r"waste\w*",
    r"ridicul\w*",
    r"unbearabl\w*",
    r"confus\w*",
    r"predictabl\w*",
    r"weak\w*",
    r"slow\w*",
    r"mess\w*",
    r"fail\w*",
    r"stupid\w*",
    r"mediocr\w*",
    r"tedious\w*",
    r"shallow\w*",
    r"forgettabl\w*",
    r"nonsens\w*",
    r"drag\w*",
    r"rush\w*",
    r"crappy\w*",
    r"annoy\w*",
    r"chees\w*",
    r"laughabl\w*",
    r"unnatural\w*",
    r"awkward\w*",
    r"embarrass\w*",
]

In [14]:
# JSON files store the test data as dictionaries
# Both the review text and the true label can be accessed from each entry
def loadjson(filename):
    with open(filename, "r") as f:
        return json.load(f)

In [15]:
# Strips each review down to a list of tokens ready for pattern matching
def prepareText(text):
  text = text.lower()
  # Remove punctuation so tokens like 'amazing!' become 'amazing'
  text = re.sub(r'[^\w\s]', '', text)
  tokens = text.split()
  # Remove stop words — words that carry no sentiment
  tokens = [token for token in tokens if token not in STOPWORDS]
  return tokens

In [16]:
def classifyReview(review):

  tokens = prepareText(review)
  pos_score = 0
  neg_score = 0
  negation_active = False

  for token in tokens:

      # If the token is a negation word, raise the flag and skip to the next token
      # The flag will flip the score of whichever sentiment keyword comes next
      if token in NEGATION_WORDS:
          negation_active = True
          continue

      matched = False

      # Check the token against every positive pattern
      for pattern in POSITIVE_PATTERNS:
          if re.fullmatch(pattern, token):
              if negation_active:
                  # Negation inverts a positive keyword into a negative score
                  neg_score += 1
              else:
                  pos_score += 1
              negation_active = False  #reset ready for next negation
              matched = True
              break

      # Only check negative patterns if no positive pattern matched
      if not matched:
          for pattern in NEGATIVE_PATTERNS:
              if re.fullmatch(pattern, token):
                  if negation_active:
                      # Negation inverts a negative keyword into a positive score
                      pos_score += 1
                  else:
                      neg_score += 1
                  negation_active = False
                  matched = True
                  break

      # If the token matched nothing, the negation window expires
      # Negation only applies to the next sentiment keyword, not the whole review
      if not matched:
          negation_active = False

  if pos_score > neg_score:
    return "positive", pos_score, neg_score
  elif pos_score < neg_score:
    return "negative", pos_score, neg_score
  else:
    return "negative", pos_score, neg_score
# If a given reviews has an equal positive and negative score, review defualt to the negative label
# This is because a mixed review typically indicate an underwhelming experience

In [17]:
# Runs the classifier across a list of reviews and prints each result
# And returns a summary of overall performance
def evaluate(reviews):
    correct = 0
    true_positive = 0  # Predicted positive AND actually positive
    false_positive = 0  # Predicted positive BUT actually negative
    false_negative = 0  # Predicted negative BUT actually positive


    example_flag = False
    for i in range(len(reviews)):

        prediction, pos_score, neg_score = classifyReview(reviews[i]["review"])
        true_label = reviews[i]["label"]
        review = reviews[i]["review"]

        # That only a single review is shown
        if example_flag == False:
          print(f"Review {i + 1}:  {review[:150]}...")
          print(f"  Prediction: {prediction} | pos_score={pos_score}, neg_score={neg_score}")
          print(f"  True Label: {true_label}")
          example_flag = True


        if prediction == true_label:
            correct += 1

        if prediction == "positive" and true_label == "positive":
            true_positive += 1
        elif prediction == "positive" and true_label == "negative":
            false_positive += 1
        elif prediction == "negative" and true_label == "positive":
            false_negative += 1

    accuracy  = correct / len(reviews)
    precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0.0
    recall    = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0.0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    print(f"\nAccuracy:  {accuracy:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall:    {recall:.2f}")
    print(f"F1:        {f1:.2f}")


In [18]:
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/Regge12/Text-classification-Positive-or-Negative-/main/reviews.json",
    "reviews.json"
)
reviews = loadjson("reviews.json")
print(f"Dataset loaded: {len(reviews)} reviews")

Dataset loaded: 65 reviews


In [20]:
# ── run_test ──────────────────────────────────────────────────────────────────
# Display helper used in all test sections
# Prints a truncated review preview, the predicted and expected labels,
# the scores, and the full reasoning trace, so every decision is explainable
# show_reasoning can be set to False for bulk output, where trace is too verbose

#Reviews 1 - 45, clearly positive or negative reviews (23 negative, 22 positive)
print("\n" + "="*20 + "Clearly negative or Positive" + "="*20)
print("These reviews show strong sentiment towards one of the binary options: positive or negative"
 + "\n The system should perform best with these reviews because of their high usage of a single classification of keywords")
print("\nFor example:")
evaluate(reviews[:45])

#Reviews 46 - 65, ambiguous reviews (mix sentiment)
print("\n" + "="*20 + "Mixed Sentiment" + "="*20)
print("These reviews show mixed sentiment towards one of the binary options: positive or negative."
 +"\nThis is difficult for the system to predict correctly because of the mix of negative and positive keywords")
print("\nFor example:")
mixedSentiment_reviews = reviews[45:]
evaluate(mixedSentiment_reviews)

# Review 6, 8, 18 - Sarcastic/ Ironic
print("\n" + "="*20 + "Sarcastic/Ironic" + "="*20)
print("These reviews show a sarcastic or ironic tone, which the system could struggle to predict correctly."
+ "\nThis is because these reviews tend to use positive keywords to express negative sentiment")
print("\nFor example:")
sarcastic_reviews = [reviews[50], reviews[52], reviews[62]]
evaluate(sarcastic_reviews)

# Reviews 7,13,14,17 - Negation
print("\n" + "="*20 + "Negation" + "="*20)
print("These reviews employ the usage of negation keywords to express an opposite sentiment to the keyword pattern match"
 + "\nThe system is build to take account of negation therefore the system should perform well")
print("\nFor example:")
negation_reviews = [reviews[51], reviews[57], reviews[58], reviews[61]]
evaluate(negation_reviews)


#Complete group of reviews
print("\n" + "="*20 + "Complete Dataset" + "="*20)
print("These reviews mimic the specturm of film reviews across the internet, \nevaluating how the system would preform on a real world dataset")
print("\nFor example:")
evaluate(reviews)


====================Clearly negative or Positive====================
These reviews show strong sentiment towards one of the binary options: positive or negative
 The system should perform best with these reviews because of their high usage of a single classification of keywords

For example:
Review 1:  One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with...
  Prediction: positive | pos_score=2, neg_score=1
  True Label: positive

Accuracy:  0.78
Precision: 0.73
Recall:    0.86
F1:        0.79

====================Mixed Sentiment====================
These reviews show mixed sentiment towards one of the binary options: positive or negative.
This is difficult for the system to predict correctly because of the mix of negative and positive keywords

For example:
Review 1:  I thought this film was visually beautiful and emotionally engaging, but the pacing was frustratingly slow in several scen